# Clase 15: Valuación de opciones europeas con GBM

Como hemos visto, la simulación de Monte Carlo es una técnica computacional que permite estimar valores esperados mediante la generación de múltiples escenarios aleatorios.

En finanzas, esta metodología se utiliza ampliamente para:

- Valuar derivados financieros
- Analizar riesgo
- Simular trayectorias de precios
- Evaluar estrategias de inversión

En el contexto de opciones financieras, Monte Carlo permite estimar el valor de una opción simulando múltiples posibles precios futuros del activo subyacente y calculando el pago esperado del contrato.

Este enfoque es particularmente útil cuando los modelos analíticos tradicionales (como Black-Scholes) no son fáciles de aplicar.

# Ventajas y limitaciones de usar Monte Carlo

## Ventajas

**Flexibilidad**

El método puede aplicarse a una gran variedad de opciones y activos subyacentes, incluso en casos donde no existe una fórmula cerrada.

**Capacidad para modelar escenarios complejos**

Permite incorporar características adicionales como:

- volatilidad estocástica
- múltiples factores de riesgo
- trayectorias dependientes del tiempo

**Facilidad conceptual**

La lógica del método es intuitiva: simular muchos posibles futuros y calcular el promedio del resultado.

## Limitaciones

**Costo computacional**

Para obtener estimaciones precisas se requieren muchas simulaciones, lo que puede aumentar el tiempo de cálculo.

**Ruido estadístico**

Los resultados dependen de la cantidad de simulaciones realizadas.

**Menor eficiencia en problemas simples**

En opciones europeas estándar, modelos como Black-Scholes pueden ser más rápidos.

# Valor intrínseco de las opciones

Esta clase calcularemos el valor de opciones europeas, es decir, opciones que solo pueden ejercerse en la fecha de vencimiento.

El pago al vencimiento (o *payoff*) es:

### Opción Call

$C_T = \max(S_T - K, 0)$

### Opción Put
$P_T = \max(K - S_T, 0)$

Donde:

- $S_T$ = precio del activo (*spot*) en el vencimiento
- $K$ = precio de ejercicio (*strike*)

---

# Valor presente de la opción

El valor actual de la opción se calcula como el **valor esperado descontado** de los pagos simulados. (El valor presente del *expected payoff*).

$C_0 = e^{-rT} \cdot \frac{1}{N} \sum_{i=1}^{N} C_{T,i}$

$P_0 = e^{-rT} \cdot \frac{1}{N} \sum_{i=1}^{N} P_{T,i}$

Donde:

- $C_0$ = valor actual de la call
- $P_0$ = valor actual de la put
- $r$ = tasa libre de riesgo
- $T$ = tiempo al vencimiento
- $N$ = número de simulaciones
- $C_{T,i}$, $P_{T,i}$ = pago en la simulación i

---

# Implementación en Python

## Importación de librerías

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

# Obtención de datos del activo

En este ejemplo utilizamos el ETF **SPY**, que replica el índice S&P 500.

In [2]:
ticker = 'SPY'
data = yf.download(ticker, start='2022-01-01', end='2026-03-10')['Close']['SPY']

[*********************100%***********************]  1 of 1 completed


# Cálculo de rendimientos y volatilidad

Primero calculamos los **rendimientos diarios** y después estimamos la **volatilidad anualizada**.

In [3]:
daily_returns = data.pct_change()
daily_returns = daily_returns.dropna()
sigma = daily_returns.std() * np.sqrt(252)

Donde:

- 252 representa el número aproximado de días de trading en un año.

# Definición de parámetros

Definimos los parámetros necesarios para la simulación.

In [4]:
S0 = data.iloc[-1]  # Precio spot actual
r = 0.08            # Tasa libre de riesgo
T = 100 / 365       # Tiempo al vencimiento en años

num_simulations = 10000

# Generación de variables aleatorias

Para simular trayectorias usamos números aleatorios provenientes de una distribución normal estándar.

In [5]:
Z = np.random.normal(0, 1, num_simulations)

# Simulación del precio futuro

Bajo el supuesto de **Movimiento Geométrico Browniano**, el precio futuro se modela como:

$S_T = S_0 e^{(r - \frac{1}{2}\sigma^2)T + \sigma\sqrt{T}Z}$

Implementación en Python:

In [6]:
St = S0 * np.exp((r - 0.5 * sigma ** 2) * T + sigma * np.sqrt(T) * Z)

**Esto genera 10,000 posibles precios del activo al vencimiento.**

---

# Cálculo de pagos de las opciones

Definimos el precio strike (precio pactado en el contrato de la opción)

In [7]:
K = 580

Calculamos el pago de la call y la put para cada simulación.

In [8]:
call = np.maximum(St - K, 0)
put = np.maximum(K - St, 0)

# Valor esperado descontado

Finalmente calculamos el valor presente de las opciones.

In [9]:
call_value = np.exp(-r * T) * np.mean(call)
put_value = np.exp(-r * T) * np.mean(put)

## Resultados

In [11]:
call_value,put_value

(np.float64(111.54058873916141), np.float64(0.6102059593018524))

Estos valores representan la estimación del precio de la opción europea call y put utilizando simulación de Monte Carlo.

Como puedes ver, la simulación de Monte Carlo permite estimar el valor de opciones financieras replicando miles de posibles escenarios futuros del activo subyacente.

**El procedimiento general consiste en:**

1. Estimar la volatilidad del activo.
2. Simular múltiples precios futuros.
3. Calcular el pago de la opción en cada escenario.
4. Promediar los resultados y descontarlos al presente.

Aunque para opciones europeas simples existen métodos analíticos más eficientes, Monte Carlo se volverá especialmente útil en problemas más complejos donde las soluciones cerradas no existen.